In [4]:
!pip uninstall -y torchvision

In [3]:
import torch
import transformers
import gradio

print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("Gradio:", gradio.__version__)
print("Environment check completed!")

PyTorch: 2.11.0+cpu
Transformers: 5.18.0
Gradio: 6.29.0
Environment check completed!


In [6]:

# EXPT NO: 10 - BUILDING A SIMPLE LLM CHATBOT
# Student Name: Shalini A
# Student Roll No: 24BAD409

import torch
import gradio as gr
from transformers import AutoTokenizer, AutoModelForCausalLM

print("EXPT NO: 10 - LLM CHATBOT")
print("Student Name: Shalini A")
print("Student Roll No: 24BAD409")

# PART A: Load conversational Transformer model
MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"

print("Loading model. Please wait...")

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype="auto",
    device_map="auto"
)

model.eval()
print("Model loaded successfully!")

# PART B: Generate a context-aware response
def generate_response(message, history):
    if not message or not message.strip():
        return "Please enter a question."

    try:
        messages = [
            {
                "role": "system",
                "content": (
                    "You are a helpful, friendly AI assistant. "
                    "Answer the user's actual question clearly and "
                    "accurately. Give useful explanations and examples "
                    "when needed. If a question is incomplete but its "
                    "meaning is clear from earlier messages, use the "
                    "conversation context. If essential information is "
                    "missing, ask a short clarifying question. Do not "
                    "invent facts. Admit uncertainty when necessary."
                )
            }
        ]

        # Keep recent conversation for follow-up questions
        for item in (history or [])[-8:]:
            if isinstance(item, dict):
                role = item.get("role")
                content = item.get("content", "")

                if role in ("user", "assistant") and isinstance(content, str):
                    messages.append({
                        "role": role,
                        "content": content
                    })

            elif isinstance(item, (list, tuple)) and len(item) == 2:
                if item[0]:
                    messages.append({
                        "role": "user",
                        "content": str(item[0])
                    })
                if item[1]:
                    messages.append({
                        "role": "assistant",
                        "content": str(item[1])
                    })

        messages.append({
            "role": "user",
            "content": message.strip()
        })

        # Apply the model's chat template
        prompt = tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True
        )

        inputs = tokenizer(
            prompt,
            return_tensors="pt",
            truncation=True,
            max_length=4096
        ).to(model.device)

        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=400,
                do_sample=False,
                repetition_penalty=1.08,
                pad_token_id=tokenizer.eos_token_id
            )

        # Decode only the newly generated answer
        new_tokens = outputs[0][inputs["input_ids"].shape[1]:]

        answer = tokenizer.decode(
            new_tokens,
            skip_special_tokens=True
        ).strip()

        return answer if answer else "Please try asking in another way."

    except Exception as e:
        return "Sorry, an error occurred: " + str(e)

# PART D: Test different prompts
test_prompts = [
    "What is Artificial Intelligence?",
    "Explain machine learning with an example.",
    "What is a Transformer in NLP?",
    "Write Python code to add two numbers.",
    "What are the advantages of chatbots?"
]

print("\nTESTING FIVE PROMPTS")

for i, prompt in enumerate(test_prompts, 1):
    print(f"\nTest {i}: {prompt}")
    print("Response:", generate_response(prompt, []))

# PART C: Create the interactive chatbot
demo = gr.ChatInterface(
    fn=generate_response,
    title="AI Chatbot - Experiment 10",
    description=(
        "Student: Shalini A | Roll No: 24BAD409\n"
        "Ask questions, request explanations, and continue "
        "the same conversation with follow-up messages."
    ),
    chatbot=gr.Chatbot(height=450),
    textbox=gr.Textbox(
        placeholder="Type your question here...",
        label="Your Message"
    )
)

# PART D: Launch the application
print("\nLaunching chatbot...")
demo.launch(share=True)


EXPT NO: 10 - LLM CHATBOT
Student Name: Shalini A
Student Roll No: 24BAD409
Loading model. Please wait...


config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

...

model.safetensors: reconstructing file:   0%|          |  0.00B / 3.09GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Model loaded successfully!

TESTING FIVE PROMPTS

Test 1: What is Artificial Intelligence?
Response: Artificial Intelligence (AI) refers to the simulation of human intelligence in machines that are programmed to think like humans and mimic their actions. These systems are called "artificial" because they lack biological brains or central nervous systems. Instead, they process data using algorithms and statistical models.

The goal of AI is to create intelligent machines that can perform tasks requiring human-like intelligence, such as learning, problem-solving, decision-making, perception, language understanding, and creativity. This field encompasses various subfields including machine learning, natural language processing, computer vision, robotics, expert systems, and more.

Key characteristics of AI include:

1. Learning: The ability to improve performance over time through experience.
2. Adaptation: The capacity to change behavior based on new inputs or feedback.
3. Reasoning: The